# Global Minimum-Variance Portfolio

This notebook examines estimation risk in mean–variance optimisation and compares maximum-Sharpe allocations with the global minimum-variance portfolio.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import risk_kit as rk

industries = rk.get_ind_returns()
estimation_returns = industries.loc["1996":"2000"]
er = rk.annualize_rets(estimation_returns, 12)
cov = estimation_returns.cov() * 12
risk_free_rate = 0.03

## 1. Maximum-Sharpe sensitivity
The covariance matrix remains fixed while expected-return assumptions are varied.

In [ ]:
assets = ["Food", "Steel"]
asset_cov = cov.loc[assets, assets]
scenarios = {
    "Historical estimates": er[assets].to_numpy(),
    "Scenario A (11%, 12%)": np.array([0.11, 0.12]),
    "Scenario B (10%, 13%)": np.array([0.10, 0.13]),
}
pd.DataFrame({name: rk.msr(risk_free_rate, values, asset_cov)
              for name, values in scenarios.items()}, index=assets)

Large changes in the maximum-Sharpe allocation despite a fixed covariance matrix indicate sensitivity to expected-return estimates.

## 2. GMV and reference portfolios

In [ ]:
frontier_assets = ["Smoke", "Fin", "Games", "Coal"]
er_n = er[frontier_assets]
cov_n = cov.loc[frontier_assets, frontier_assets]
ax = rk.plot_ef(60, er_n, cov_n, riskfree_rate=risk_free_rate,
                show_cml=True, show_ew=True, show_gmv=True)
ax.set_title("Efficient frontier, GMV and maximum-Sharpe portfolios")
plt.tight_layout()

In [ ]:
gmv_weights = rk.gmv(cov_n)
pd.Series(gmv_weights, index=frontier_assets, name="GMV allocation")

The GMV portfolio depends on covariance estimates rather than expected-return estimates. It is not free of estimation risk: covariance estimates also depend on the sample period and data quality.

## Limitations
The risk-free rate is illustrative. The analysis is historical and in-sample, long-only, and does not include transaction costs, taxes, liquidity constraints or out-of-sample execution.